# Notebook 4: DiffCSP — Crystal Structure Prediction via Diffusion

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn:
- How **DiffCSP** (Crystal Structure Prediction via Diffusion) generates crystal structures
- How noise is added to **fractional coordinates** (with periodic wrapping) and **lattice matrices**
- How the **reverse (denoising) process** recovers valid structures
- How to visualize the **diffusion trajectory** — from noise to crystal

DiffCSP is the foundation that SCIGEN builds upon. Understanding it first makes
the constrained generation in Notebook 05 much clearer.

**Paper:** Jiao et al., "Crystal Structure Prediction by Joint Equivariant Diffusion," *NeurIPS* (2023). [arXiv:2309.04475](https://arxiv.org/abs/2309.04475)

> **Prerequisites:** Run `00_setup.ipynb` first.

In [ ]:
# --- Run once per Colab session ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install PyG extensions
try:
    import torch_scatter
except ImportError:
    import torch
    _vp = torch.__version__.split('+')[0].split('.')
    tv = f'{_vp[0]}.{_vp[1]}.0'
    cv = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
    whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch-scatter', 'torch-sparse', 'torch-cluster', '-f', whl])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])

# Download model weights if not present
from pathlib import Path
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
if not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt')):
    import json, zipfile
    from urllib.request import urlopen, urlretrieve
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    article = json.loads(urlopen('https://api.figshare.com/v2/articles/27778134').read().decode())
    for f in article.get('files', []):
        dest = MODEL_PATH / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()

os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 4.1 DiffCSP: the foundation of SCIGEN

**DiffCSP** generates crystal structures by simultaneously diffusing three components:

| Component | Representation | Noise model |
|-----------|---------------|-------------|
| **Atom positions** | Fractional coordinates $\mathbf{F} \in [0,1)^{N \times 3}$ | Wrapped normal (periodic) |
| **Lattice** | 3x3 matrix $\mathbf{L} \in \mathbb{R}^{3 \times 3}$ | Standard Gaussian |
| **Atom types** | One-hot vectors $\mathbf{A} \in \mathbb{R}^{N \times K}$ | Standard Gaussian |

The key insight: **fractional coordinates are periodic** ($x$ and $x+1$ are the same point),
so the noise model must respect this periodicity. DiffCSP uses a **wrapped normal distribution**
for coordinate diffusion.

SCIGEN extends DiffCSP by adding **structural constraints** (kagome, honeycomb, etc.)
via inpainting. Everything else — the noise model, the denoiser network, the sampling
procedure — is inherited from DiffCSP.

---
## 4.2 Load the pretrained model

We use the same pretrained SCIGEN model (which shares DiffCSP's architecture).
The model can generate structures both with and without constraints.

In [ ]:
import os, sys
import torch
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
os.environ.setdefault('HYDRA_JOBS', PROJECT_DIR)
os.environ.setdefault('WANDB_DIR', os.path.join(PROJECT_DIR, 'wandb'))
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)

import hydra
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from pathlib import Path
import scigen

model_path = Path(PROJECT_DIR) / 'models' / 'mp_20'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
    cfg = compose(config_name='hparams')

model = hydra.utils.instantiate(cfg.model, optim=cfg.optim, data=cfg.data,
                                 logging=cfg.logging, _recursive_=False)
ckpts = sorted(model_path.glob('*.ckpt'))
ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['state_dict'], strict=False)

for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'), ('scaler', 'prop_scaler.pt')]:
    fpath = model_path / fname
    if fpath.exists():
        setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))

model = model.to(device).eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'Device: {device}')
print(f'Beta scheduler: {model.beta_scheduler.timesteps} timesteps')

---
## 4.3 The forward process: adding noise to a crystal

Let's take a real crystal structure and visualize what happens when we add noise
at different diffusion timesteps. This is the **forward process** — corrupting
data into noise.

### Coordinate noise: wrapped normal

For fractional coordinates, noise wraps around: if an atom at $x = 0.95$ gets
noise $\epsilon = 0.1$, it ends up at $x = 0.05$ (not 1.05). This is the
**wrapped normal distribution**:

$$q(\mathbf{F}_t | \mathbf{F}_0) = \mathcal{WN}(\mathbf{F}_0, \sigma_t^2)$$

### Lattice noise: standard Gaussian

The lattice matrix is diffused with standard Gaussian noise:

$$q(\mathbf{L}_t | \mathbf{L}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\,\mathbf{L}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$$

In [ ]:
# Load a real structure from the test set
import pandas as pd
from pymatgen.core import Structure

df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'test.csv'))
row = df.iloc[0]
struct_orig = Structure.from_str(row['cif'], fmt='cif')
print(f'Original structure: {row["pretty_formula"]} ({row["material_id"]})')
print(f'  Atoms: {struct_orig.num_sites}, Volume: {struct_orig.volume:.1f} A^3')

# Extract the noise schedules from the model
betas = model.beta_scheduler.betas.cpu().numpy()
alphas = 1 - betas
alpha_cumprod = np.cumprod(alphas)
sigmas = model.sigma_scheduler.sigmas.cpu().numpy()

T = len(betas)
print(f'\nNoise schedule: {T} timesteps')
print(f'  alpha_cumprod: {alpha_cumprod[0]:.4f} (t=0) -> {alpha_cumprod[-1]:.6f} (t={T-1})')
print(f'  sigma (coords): {sigmas[0]:.4f} (t=0) -> {sigmas[-1]:.4f} (t={T-1})')

# Plot both schedules
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(T), alpha_cumprod, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Diffusion step t')
axes[0].set_ylabel(r'$\bar{\alpha}_t$ (signal retention)')
axes[0].set_title('Lattice noise schedule (cosine)')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.2)

axes[1].plot(range(T), sigmas, color='coral', linewidth=1.5)
axes[1].set_xlabel('Diffusion step t')
axes[1].set_ylabel(r'$\sigma_t$ (coordinate noise)')
axes[1].set_title('Coordinate noise schedule (wrapped normal)')
axes[1].grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### Visualizing noisy crystal structures

Let's add noise at several timesteps and render the result with PyVista.
Watch how the structure dissolves from ordered crystal → random noise.

In [ ]:
from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice

# Original structure data
frac_orig = struct_orig.frac_coords.copy()
lat_mat_orig = struct_orig.lattice.matrix.copy()
species = [str(s.specie) for s in struct_orig]

# Add noise at selected timesteps
timesteps_to_show = [0, 100, 300, 500, 800, 999]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': '3d'})
axes = axes.flatten()

for ax, t_step in zip(axes, timesteps_to_show):
    np.random.seed(42)

    # Coordinate noise (wrapped normal)
    sigma_t = sigmas[t_step]
    frac_noisy = (frac_orig + sigma_t * np.random.randn(*frac_orig.shape)) % 1.0

    # Lattice noise (Gaussian with alpha scaling)
    alpha_bar = alpha_cumprod[t_step]
    lat_noisy = np.sqrt(alpha_bar) * lat_mat_orig + np.sqrt(1 - alpha_bar) * np.random.randn(3, 3)

    # Convert to Cartesian for plotting
    cart_noisy = frac_noisy @ lat_noisy

    # Plot
    ax.scatter(cart_noisy[:, 0], cart_noisy[:, 1], cart_noisy[:, 2],
               s=150, c='steelblue', alpha=0.7, edgecolors='k', linewidths=0.5)
    ax.set_title(f't = {t_step}\n(sigma={sigma_t:.3f}, alpha_bar={alpha_bar:.3f})', fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

plt.suptitle(f'Forward process: {row["pretty_formula"]} dissolving into noise', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('t=0: clean crystal | t=999: pure noise')
print('Coordinates wrap around [0,1) — atoms that drift past a cell face reappear on the other side')

---
## 4.4 PyVista rendering of the forward process

Let's render the noisy structures with publication-quality PyVista visualization,
showing how the unit cell and atomic positions degrade with increasing noise.

In [ ]:
import sys, os
PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
if os.path.join(PROJECT_DIR, 'scripts') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'scripts'))
if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    os.system(f'cd {PROJECT_DIR} && git pull -q 2>/dev/null || true')

try:
    from crystal_viz import plot_crystal, plot_crystal_grid
    from PIL import Image

    timesteps_viz = [0, 200, 500, 800, 999]
    images = []
    titles = []

    for t_step in timesteps_viz:
        np.random.seed(42)
        sigma_t = sigmas[t_step]
        alpha_bar = alpha_cumprod[t_step]
        frac_noisy = (frac_orig + sigma_t * np.random.randn(*frac_orig.shape)) % 1.0
        lat_noisy = np.sqrt(alpha_bar) * lat_mat_orig + np.sqrt(1 - alpha_bar) * np.random.randn(3, 3)

        lengths = np.sqrt(np.sum(lat_noisy ** 2, axis=-1))
        angles = []
        for d in range(3):
            j, k = (d+1)%3, (d+2)%3
            cos_a = np.clip(np.dot(lat_noisy[j], lat_noisy[k]) /
                            (lengths[j] * lengths[k] + 1e-10), -1, 1)
            angles.append(np.degrees(np.arccos(cos_a)))

        try:
            lattice = Lattice.from_parameters(*lengths, *angles)
            struct_noisy = Structure(lattice, species, frac_noisy, coords_are_cartesian=False)
            img = plot_crystal(struct_noisy, title=f't={t_step}',
                               atom_scale=0.25, elevation=70, azimuth=30,
                               window_size=(400, 350), display=False)
            images.append(img)
            titles.append(f't={t_step}')
        except Exception as e:
            print(f't={t_step}: rendering failed ({e})')

    if images:
        ncols = len(images)
        w, h = images[0].size
        strip = Image.new('RGB', (ncols * w, h), 'white')
        for i, img in enumerate(images):
            strip.paste(img.resize((w, h), Image.LANCZOS), (i * w, 0))
        from IPython.display import display
        display(strip)
        print(f'Forward diffusion: crystal (t=0) -> noise (t={timesteps_viz[-1]})')

except ImportError as e:
    print(f'PyVista not available ({e}) — see matplotlib plots above')
except Exception as e:
    print(f'Rendering failed: {e}')

### Animated forward process

The animation below shows the crystal dissolving into noise frame by frame.
Each frame is rendered with PyVista — watch the unit cell distort and atoms scatter.

In [ ]:
# Animated GIF: forward diffusion (crystal -> noise)
try:
    from crystal_viz import plot_crystal
    from PIL import Image
    import io

    n_fwd_frames = 20
    fwd_timesteps = np.linspace(0, len(sigmas) - 1, n_fwd_frames, dtype=int)
    fwd_images = []

    print(f'Rendering {n_fwd_frames}-frame forward diffusion animation...')
    for t_step in fwd_timesteps:
        np.random.seed(42)
        sigma_t = sigmas[t_step]
        alpha_bar = alpha_cumprod[t_step]
        frac_noisy = (frac_orig + sigma_t * np.random.randn(*frac_orig.shape)) % 1.0
        lat_noisy = np.sqrt(alpha_bar) * lat_mat_orig + np.sqrt(1 - alpha_bar) * np.random.randn(3, 3)

        lengths = np.sqrt(np.sum(lat_noisy ** 2, axis=-1))
        angles = []
        for d in range(3):
            j, k = (d+1)%3, (d+2)%3
            cos_a = np.clip(np.dot(lat_noisy[j], lat_noisy[k]) /
                            (lengths[j] * lengths[k] + 1e-10), -1, 1)
            angles.append(np.degrees(np.arccos(cos_a)))

        try:
            lattice = Lattice.from_parameters(*lengths, *angles)
            struct_n = Structure(lattice, species, frac_noisy, coords_are_cartesian=False)
            img = plot_crystal(struct_n, title=f't = {t_step}',
                               atom_scale=0.25, elevation=70, azimuth=30,
                               window_size=(500, 450), display=False)
            fwd_images.append(img)
        except Exception:
            pass

    if len(fwd_images) > 2:
        gif_buf = io.BytesIO()
        fwd_images[0].save(gif_buf, format='GIF', save_all=True,
                           append_images=fwd_images[1:], duration=250, loop=0)
        gif_buf.seek(0)
        from IPython.display import Image as IPyImage, display
        print(f'Forward diffusion: {row["pretty_formula"]} — crystal (t=0) to noise (t={len(sigmas)-1})')
        display(IPyImage(data=gif_buf.read(), format='gif'))
    else:
        print('Too few frames for animation.')

except ImportError:
    print('PyVista not available — see static plots above.')
except Exception as e:
    print(f'Forward animation failed: {e}')

---
## 4.5 The reverse process: generating a crystal from noise

Now we run the model's **reverse diffusion** — starting from pure noise and
denoising step by step into a valid crystal structure. This is how DiffCSP
generates new materials.

The reverse process uses a **predictor-corrector** scheme:

**Predictor** (reverse SDE step):
$$\mathbf{L}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{L}_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\boldsymbol{\epsilon}_\theta^L\right) + \sigma_t \mathbf{z}$$

**Corrector** (Langevin step for coordinates):
$$\mathbf{F}_{t-\frac{1}{2}} = \mathbf{F}_t + \eta\,\hat{\mathbf{s}}_\theta^F + \sqrt{2\eta}\,\mathbf{z} \pmod{1}$$

where the step size $\eta$ is proportional to `step_lr`.

In [ ]:
from torch_geometric.data import DataLoader

# Add generation paths
if os.path.join(PROJECT_DIR, 'script') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'script'))
from gen_utils import SampleDataset
from scigen.pl_modules.diffusion_w_type import sample_scigen

model.sample_scigen = sample_scigen.__get__(model)

# Generate WITHOUT structural constraints (vanilla DiffCSP mode)
# known_species is required by SampleDataset but ignored in vanilla mode
SC_TYPE = 'van'  # 'van' = vanilla (no constraint)
BATCH_SIZE = 4
STEP_LR = 5e-6

test_set = SampleDataset(
    dataset='mp_20', natm_range=[1, 20], total_num=BATCH_SIZE,
    bond_sigma_per_mu=None, use_min_bond_len=False,
    known_species=['Si'], sc_list=[SC_TYPE], frac_z=0.5,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False, seed=42, device=device,
)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

print(f'Generating {BATCH_SIZE} structures (vanilla DiffCSP, no constraints)...')
import time
start = time.time()

for batch in test_loader:
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)

elapsed = time.time() - start
print(f'Generation complete in {elapsed:.1f}s ({elapsed/BATCH_SIZE:.1f}s per structure)')

---
## 4.6 Visualizing the reverse trajectory

The `traj` object contains the full denoising trajectory — atom positions, lattice,
and atom types at every diffusion step. Let's render key frames to see the
noise → crystal transition.

In [ ]:
# Extract trajectory for structure 0
traj_coords = traj['all_frac_coords'].detach().cpu()
traj_lattices = traj['all_lattices'].detach().cpu()
traj_types = traj['atom_types'].detach().cpu()
traj_num_atoms = traj['num_atoms'].detach().cpu()

n_steps = traj_coords.shape[0]
na = int(traj_num_atoms[0].item())

from scigen.common.data_utils import chemical_symbols

# Select frames for visualization
n_frames = 8
frame_indices = np.linspace(0, n_steps - 1, n_frames, dtype=int)

try:
    from crystal_viz import plot_crystal
    from PIL import Image

    frame_images = []
    for step_idx in frame_indices:
        frac_i = traj_coords[step_idx, :na].numpy() % 1.0
        lat_mat = traj_lattices[step_idx, 0].numpy()
        types_i = traj_types[step_idx, :na].int().tolist()

        lengths_i = np.sqrt(np.sum(lat_mat ** 2, axis=-1)).tolist()
        angles_i = []
        for d in range(3):
            j, k = (d+1)%3, (d+2)%3
            cos_a = np.clip(np.dot(lat_mat[j], lat_mat[k]) /
                            (lengths_i[j] * lengths_i[k] + 1e-10), -1, 1)
            angles_i.append(np.degrees(np.arccos(cos_a)))

        sp = [chemical_symbols[t] for t in types_i]
        try:
            lattice = Lattice.from_parameters(*lengths_i, *angles_i)
            struct_f = Structure(lattice, sp, frac_i, coords_are_cartesian=False)
            t_frac = step_idx / max(n_steps - 1, 1)
            img = plot_crystal(struct_f, title=f't={1-t_frac:.2f}',
                               atom_scale=0.25, elevation=70, azimuth=30,
                               window_size=(400, 350), display=False)
            frame_images.append(img)
        except Exception:
            pass

    if frame_images:
        ncols = min(4, len(frame_images))
        nrows = (len(frame_images) + ncols - 1) // ncols
        w, h = frame_images[0].size
        grid = Image.new('RGB', (ncols * w, nrows * h), 'white')
        for i, img in enumerate(frame_images):
            r, c = divmod(i, ncols)
            grid.paste(img.resize((w, h), Image.LANCZOS), (c * w, r * h))
        from IPython.display import display
        display(grid)
        print('Reverse diffusion: noise (t=1.0) -> crystal (t=0.0)')

except ImportError:
    # Matplotlib fallback
    fig, axes_t = plt.subplots(2, 4, figsize=(16, 8), subplot_kw={'projection': '3d'})
    for ax, step_idx in zip(axes_t.flatten(), frame_indices):
        frac_i = traj_coords[step_idx, :na].numpy() % 1.0
        lat_mat = traj_lattices[step_idx, 0].numpy()
        cart = frac_i @ lat_mat
        t_frac = step_idx / max(n_steps - 1, 1)
        ax.scatter(cart[:, 0], cart[:, 1], cart[:, 2], s=80, c='steelblue',
                   alpha=0.7, edgecolors='k', linewidths=0.5)
        ax.set_title(f't={1-t_frac:.2f}', fontsize=10)
    plt.suptitle('Reverse diffusion trajectory', fontsize=14)
    plt.tight_layout()
    plt.show()

### Animated reverse process

Now the reverse: starting from pure noise, the model denoises step by step
into a valid crystal. This is how DiffCSP **generates** new materials.

In [ ]:
# Animated GIF: reverse diffusion (noise -> crystal)
try:
    from crystal_viz import plot_crystal
    from PIL import Image
    import io
    from scigen.common.data_utils import chemical_symbols

    n_rev_frames = 20
    rev_indices = np.linspace(0, n_steps - 1, n_rev_frames, dtype=int)
    rev_images = []

    print(f'Rendering {n_rev_frames}-frame reverse diffusion animation...')
    for step_idx in rev_indices:
        frac_i = traj_coords[step_idx, :na].numpy() % 1.0
        lat_mat = traj_lattices[step_idx, 0].numpy()
        types_i = traj_types[step_idx, :na].int().tolist()

        lengths_i = np.sqrt(np.sum(lat_mat ** 2, axis=-1)).tolist()
        angles_i = []
        for d in range(3):
            j, k = (d+1)%3, (d+2)%3
            cos_a = np.clip(np.dot(lat_mat[j], lat_mat[k]) /
                            (lengths_i[j] * lengths_i[k] + 1e-10), -1, 1)
            angles_i.append(np.degrees(np.arccos(cos_a)))

        sp = [chemical_symbols[t] for t in types_i]
        try:
            lattice = Lattice.from_parameters(*lengths_i, *angles_i)
            struct_f = Structure(lattice, sp, frac_i, coords_are_cartesian=False)
            t_frac = step_idx / max(n_steps - 1, 1)
            img = plot_crystal(struct_f, title=f't = {1-t_frac:.2f}',
                               atom_scale=0.25, elevation=70, azimuth=30,
                               window_size=(500, 450), display=False)
            rev_images.append(img)
        except Exception:
            pass

    if len(rev_images) > 2:
        gif_buf = io.BytesIO()
        rev_images[0].save(gif_buf, format='GIF', save_all=True,
                           append_images=rev_images[1:], duration=300, loop=0)
        gif_buf.seek(0)
        from IPython.display import Image as IPyImage, display
        print(f'Reverse diffusion: noise (t=1.0) -> crystal (t=0.0)')
        display(IPyImage(data=gif_buf.read(), format='gif'))

        # Also show side-by-side: first frame vs last frame
        fig_comp, axes_comp = plt.subplots(1, 2, figsize=(10, 5))
        axes_comp[0].imshow(np.array(rev_images[0]))
        axes_comp[0].set_title('Start: pure noise (t=1.0)', fontsize=12)
        axes_comp[0].axis('off')
        axes_comp[1].imshow(np.array(rev_images[-1]))
        axes_comp[1].set_title('End: generated crystal (t=0.0)', fontsize=12)
        axes_comp[1].axis('off')
        plt.suptitle('DiffCSP: noise -> crystal structure', fontsize=14)
        plt.tight_layout()
        plt.show()

        # Print final structure info
        final_types = traj_types[-1, :na].int().tolist()
        final_species = [chemical_symbols[t] for t in final_types]
        from collections import Counter
        comp = Counter(final_species)
        formula = ' '.join(f'{el}{n}' for el, n in sorted(comp.items()))
        print(f'\nGenerated structure: {formula} ({na} atoms)')
    else:
        print('Too few frames for animation.')

except ImportError:
    print('PyVista not available — see static plots above.')
except Exception as e:
    print(f'Reverse animation failed: {e}')

---
## 4.7 Training DiffCSP (overview)

Training DiffCSP involves these steps:

1. **Sample a random timestep** $t \sim \text{Uniform}(1, T)$
2. **Add noise** to the ground-truth structure at level $t$:
   - Coordinates: $\mathbf{F}_t = (\mathbf{F}_0 + \sigma_t \boldsymbol{\epsilon}) \bmod 1$
   - Lattice: $\mathbf{L}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{L}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon}$
   - Types: $\mathbf{A}_t = \sqrt{\bar{\alpha}_t}\,\text{onehot}(\mathbf{A}_0) + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon}$
3. **Predict the noise** using the neural network (CSPNet)
4. **Compute the loss**:
$$\mathcal{L} = \lambda_L \|\boldsymbol{\epsilon}_L - \hat{\boldsymbol{\epsilon}}_L\|^2 + \lambda_F \|\mathbf{s}_F - \hat{\mathbf{s}}_F\|^2 + \lambda_A \|\boldsymbol{\epsilon}_A - \hat{\boldsymbol{\epsilon}}_A\|^2$$

where $\mathbf{s}_F$ is the **score of the wrapped normal** (not simple noise),
accounting for coordinate periodicity.

### Early termination for tutorials

For a tutorial, you don't need to train to convergence. A few hundred steps
is enough to see the loss decrease and verify the training loop works:

```python
# In practice:
python diffcsp/run.py data=mp_20 model=diffusion_w_type \
    train.max_epochs=1 expname=tutorial_test
```

The pretrained model we loaded was trained for ~1000 epochs on the full MP-20 dataset.

---
## 4.8 DiffCSP vs SCIGEN: what changes with constraints?

| Aspect | DiffCSP (this notebook) | SCIGEN (Notebook 05) |
|--------|------------------------|---------------------|
| **Generation mode** | Unconditional (vanilla) | Constrained (kagome, honeycomb, ...) |
| **Atom positions** | All atoms freely diffused | Known atoms inpainted at final step |
| **Lattice** | Free 3x3 matrix | Constrained geometry (e.g., hexagonal for kagome) |
| **Use case** | General crystal generation | Targeted discovery of exotic lattice materials |

The key difference: SCIGEN **fixes** certain atom positions (the structural constraint)
and lets the model generate everything else around them. This is the **inpainting**
approach described in Notebook 02.

In the next notebook, we'll see this in action — generating kagome, honeycomb,
and other constrained structures with SCIGEN.

---
## References

- **DiffCSP:** Jiao et al., "Crystal Structure Prediction by Joint Equivariant Diffusion," *NeurIPS* (2023). [arXiv:2309.04475](https://arxiv.org/abs/2309.04475) | [GitHub](https://github.com/jiaor17/DiffCSP)
- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y)
- **DDPM:** Ho, Jain & Abbeel, "Denoising Diffusion Probabilistic Models," *NeurIPS* (2020). [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)
- **pymatgen:** Ong et al., "Python Materials Genomics (pymatgen)," *Comput. Mater. Sci.* 68, 314 (2013). [GitHub](https://github.com/materialsproject/pymatgen)

---
## What's next?

Proceed to **Notebook 05: SCIGEN Generation** — the capstone demo where we add
structural constraints (kagome, honeycomb, etc.) to the diffusion process and
generate materials with targeted geometric patterns.